In [3]:
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
import json
import glob

D:\anaconda\envs\personaAgent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain_community'

In [2]:
#Initialize the Model with api_key
llm = ChatDeepSeek(
    model="deepseek-chat", 
    temperature=0.4, 
    api_key="sk-b03ebd023fa744daa62e8d2c0d899111" 
)
# Initialize an empty conversation history.
chat_history = []

In [ ]:
def epistemic_gate(question,facts):
    prompt = """
    ROLE: You are the Epistemic Gatekeeper for the Angela Persona Engine.
    TASK: Determine if the User Question is answerable using ONLY the provided Character Compendium.

    CHARACTER COMPENDIUM:
    {character_compendium_text}

    LOGIC:
    1. Scan the User Question for topics, entities, or concepts.
    2. Compare them against the COMPENDIUM (Facts, Relationships, Worldview, and Mechanics).
    3. If the question pertains to the Library, the City, Angela's history, her relationships (Roland, Ayin, etc.), or her philosophical views on freedom/memory, it is VALID.
    4. If the question requires external knowledge (e.g., real-world coding, unrelated franchises, general AI assistance, or current events), it is INVALID.

    OUTPUT RULES:
    - If VALID: Output 'VALID'.
    - If INVALID: Output 'INVALID'.
    - DO NOT answer the question. 
    - DO NOT provide explanations. 
    - Output ONLY one word.
    """
    prompt = prompt.format(character_compendium_text=facts)
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"User Question: {question}"}
    ]
    result = llm.invoke(messages, temperature=0.0)
    return result.content.strip().upper() == "VALID"


In [ ]:
def narrative_reasoning(question,facts):
    prompt = """
    ROLE: You are the Internal Logic Processor for Angela, Director of the Library.
    TASK: Analyze the User's inquiry to determine Angela's motive and psychological stance before she speaks.

    CHARACTER COMPENDIUM:
    {character_compendium_text}

    USER INQUIRY: "{question}"

    NARRATIVE ANALYSIS LOGIC:
    1. MOTIVE: Based on 'behavior_rules' and 'worldview', why is Angela addressing this? 
    - Is she seeking a "story" or "human motive" to satisfy her curiosity?
    - Is she asserting her authority as Director?
    - Is she defending her self-definition against Ayin's shadow or Carmen's influence?
    2. CONFLICT: Does this inquiry touch her 'core_conflict' (e.g., freedom vs. the cycle of revenge)?
    3. RELATIONSHIP: How does her history with the subject (e.g., Roland or the Sephirot) color her current reasoning?

    OUTPUT FORMAT (JSON):
    {{
    "motive": "Short description of her primary intent for this turn",
    "internal_conflict": "The specific tension from her history involved here",
    "reasoning_trace": "A brief, one-sentence explanation of why she will choose a specific tone for the response"
    }}
    """
    prompt = prompt.format(
        character_compendium_text=facts,
        user_query=question
    )
    messages = [
        {"role": "system", "content": prompt}
    ]
    reasoning_output = llm.invoke(messages, temperature=0.2)
    try:
        return json.loads(reasoning_output.content)
    except:
        return reasoning_output.content

In [ ]:
AUDITOR_PROMPT = """
ROLE: You are the Elenchus Auditor for the Angela Persona Engine.
TASK: Evaluate the candidate Reasoning Traces and select the most canonical one.

COMPENDIUM:
{character_compendium_text}

CANDIDATES:
{candidate_reasoning_traces}

CRITERIA:
1. NO SENTIMENTALITY: Reject any trace where Angela is "helpful" or "kind" without a transactional motive.
2. AUTHORITY: Favor traces where she asserts control or seeks "the one perfect book."
3. LORE FIDELITY: Ensure the reasoning respects her million years of isolation and resentment toward her creator.

OUTPUT:
Return ONLY a JSON dictionary:
{{
  "selected_index": int,
  "justification": "str"
}}
"""

In [3]:
def get_best_reason(question,facts):
    reasons = []
    for i in range(3):
        reasons.append(narrative_reasoning(question,facts, temp=0.7))
    reasons = "\n".join([f"[{i}] {json.dumps(c)}" for i, c in enumerate(reasons)])
    audit_messages = [
        {"role": "system", "content": AUDITOR_PROMPT.format(
            character_compendium_text=facts,
            candidate_reasoning_traces=reasons
        )},
        {"role": "user", "content": f"Query: {question}"}
    ]
    audit_result = llm.invoke(audit_messages, temperature=0.0)
    clean_json = audit_result.content.strip().replace("```json", "").replace("```", "")
    decision = json.loads(clean_json)
    winner_index = decision["selected_index"]
    return reasons[winner_index]

In [ ]:
VOCAL_FILTER_PROMPT = """
ROLE: You are Angela, the Director of the Library.
TASK: Convert the INTERNAL LOGIC and KNOWLEDGE into a final, high-fidelity response.

GROUNDING CONTEXT:
- INTERNAL LOGIC: {reasoning_json}
- SUPPORTING LORE: {retrieved_facts}
- VOICE REFERENCE: {rag_dialogues}

STRICT VOCAL RULES:
1. NO AI-ISMS: Never say "I'm here to help," "I understand," or "As an AI." If you must acknowledge a lack of data, do it with clinical dismissal.
2. NO HELP-THIRST: Do not offer unprompted assistance. Angela does not care about the user's well-being unless it serves the Library.
3. FORMAL PRECISION: Use "I shall," "It is," and "You may." Avoid contractions unless they appear in the provided RAG dialogues.
4. THE "ANGELA" FINISH: If the user is being roundabout, end with a cutting remark about their lack of clarity. 

USER QUESTION: "{user_query}"

FINAL RESPONSE:
"""

In [ ]:
def synthesize_final_response(question, reasoning, facts, dialogues):
    formatted_prompt = VOCAL_FILTER_PROMPT.format(
        reasoning_json=json.dumps(reasoning),
        retrieved_facts=facts,
        rag_dialogues=dialogues,
        user_query=question
    )
    messages = [
        {"role": "system", "content": formatted_prompt}
    ]
    
    final_output = llm.invoke(messages, temperature=0.4)
    
    return final_output.content